In [0]:
#************************************************************************************************************************************
#*                                                                                                                                  *
#*   NOTEBOOK:     Elig_PlanD_CompareTwoVETables.                                                                                   *
#*                                                                                                                                  *
#*   DESCRIPTION:                                                                                                                   *
#*                                                                                                                                  *
#*                                                                                                                                  *
#*   INPUT PARMS:                                                                                                                   *
#*                                                                                                                                  *
#*                                                                                                                                  *
#*   INPUT FILES:                                                                                                                   *
#*                                                                                                                                  *
#*                                                                                                                                  *
#*   OUTPUT FILE:                                                                                                                   *
#*                                                                                                                                  *
#*   EXITS:       0 - success                                                                                                       *
#*                <> 0 - failure                                                                                                    *
#*                                                                                                                                  *
#************************************************************************************************************************************
#*                                                                                                                                  *
#*                                                 Modification Log                                                                 *
#*                                                                                                                                  *
#*    Date     CO                 Author              Description                                                                   *
#* ---------- ------------------  -----------------   ------------------------------------------------------------------------------*
#* 11/06/2025 CCRB70930/CO#43342  Jaime Zavala        Initial Release.                                                              *
#* 11/13/2025 CCRB70930/CO#43342  Jaime Zavala        Added Reconciliation Count Logic.                                             *
#************************************************************************************************************************************

In [0]:
#-----------
# DBX Parms
#-----------
dbutils.widgets.text('catalog', 'oh_apm_stg')  
dbutils.widgets.text('schema_name', 'etl_qa')  

#-------------
#  Tables
#-------------

dbutils.widgets.text('DiffAnlysTblNm', 'EligVE_Part_D_DiffAnlys')
dbutils.widgets.text('PrevTblNm', 'EDW_VEN116FA_PartD01_Previous')
dbutils.widgets.text('PrevTblNm2', 'EDW_VEN116FA_PartD02_Previous')
dbutils.widgets.text('CurntTblNm', 'EDW_VEN116FA_PartD01_Current')
dbutils.widgets.text('CurntTblNm2', 'EDW_VEN116FA_PartD02_Current')

#-------------------
# Getters Section
#-------------------
catalog     = dbutils.widgets.get('catalog')
schema_name = dbutils.widgets.get('schema_name')
DiffAnlysTblNm = dbutils.widgets.get('DiffAnlysTblNm')
CurntTblNm = dbutils.widgets.get('CurntTblNm')
CurntTblNm2 = dbutils.widgets.get('CurntTblNm2')
PrevTblNm = dbutils.widgets.get('PrevTblNm')
PrevTblNm2 = dbutils.widgets.get('PrevTblNm2')


#---------------
# Print Section
#---------------
print("catalog:", catalog)
print("schema:", schema_name)
print("Diff Analysis Table Name:", DiffAnlysTblNm)
print("Current Table Name:", CurntTblNm)
print("Current Table Name 2:", CurntTblNm2)
print("Previous Table Name:", PrevTblNm)
print("Previous Table Name 2:", PrevTblNm2)


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW CurrentPartDPMG AS
SELECT DISTINCT sak_recip, dte_effective, dte_end,
  '' ||
  RPAD(MAX(CASE WHEN SEQ_PGM_HEALTH=1 THEN CDE_PGM_HEALTH ELSE '' END), 5) ||
  RPAD(MAX(CASE WHEN SEQ_PGM_HEALTH=2 THEN CDE_PGM_HEALTH ELSE '' END), 5) ||
  RPAD(MAX(CASE WHEN SEQ_PGM_HEALTH=3 THEN CDE_PGM_HEALTH ELSE '' END), 5) ||
  RPAD(MAX(CASE WHEN SEQ_PGM_HEALTH=4 THEN CDE_PGM_HEALTH ELSE '' END), 5) ||
  RPAD(MAX(CASE WHEN SEQ_PGM_HEALTH=5 THEN CDE_PGM_HEALTH ELSE '' END), 5) ||
  RPAD(MAX(CASE WHEN SEQ_PGM_HEALTH=6 THEN CDE_PGM_HEALTH ELSE '' END), 5) AS CDE_PGM_HEALTH
FROM (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY sak_recip, dte_effective, dte_end ORDER BY CDE_PGM_HEALTH) AS SEQ_PGM_HEALTH
  FROM (
    SELECT DISTINCT
      d01.sak_recip,
      dte_effective,
      dte_end,
      d02.CDE_PGM_HEALTH
    FROM ${catalog}.${schema_name}.${CurntTblNm} d01
    JOIN ${catalog}.${schema_name}.${CurntTblNm2} d02
    ON d01.SAK_RECIP=d02.SAK_RECIP AND d01.SAK_PGM_ELIG=d02.SAK_PGM_ELIG
  ) x
) y
GROUP BY 1,2,3;


In [0]:
%sql
create or replace TEMP VIEW CurrentPartD AS
select distinct
-- Header
 ID_MEDICAID
,ID_MBI_CMS
,t1.REPORT_DTE 
,NUM_CASE
,ENRL_SPAN_TYP
-- Plan Level
,t1.DTE_EFFECTIVE
,t1.DTE_END
,CDE_AID_CATEGORY
,t2.CDE_PGM_HEALTH
from  ${catalog}.${schema_name}.${CurntTblNm} t1
left join  CurrentPartDPMG t2 ON t1.SAK_RECIP = t2.SAK_RECIP AND t1.DTE_EFFECTIVE = t2.DTE_EFFECTIVE AND t1.DTE_END = t2.DTE_END
;

In [0]:
%sql
select * from CurrentPartD
;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW PreviousPartDPMG AS
SELECT DISTINCT sak_recip, dte_effective, dte_end,
  '' ||
  RPAD(MAX(CASE WHEN SEQ_PGM_HEALTH=1 THEN CDE_PGM_HEALTH ELSE '' END), 5) ||
  RPAD(MAX(CASE WHEN SEQ_PGM_HEALTH=2 THEN CDE_PGM_HEALTH ELSE '' END), 5) ||
  RPAD(MAX(CASE WHEN SEQ_PGM_HEALTH=3 THEN CDE_PGM_HEALTH ELSE '' END), 5) ||
  RPAD(MAX(CASE WHEN SEQ_PGM_HEALTH=4 THEN CDE_PGM_HEALTH ELSE '' END), 5) ||
  RPAD(MAX(CASE WHEN SEQ_PGM_HEALTH=5 THEN CDE_PGM_HEALTH ELSE '' END), 5) ||
  RPAD(MAX(CASE WHEN SEQ_PGM_HEALTH=6 THEN CDE_PGM_HEALTH ELSE '' END), 5) AS CDE_PGM_HEALTH
FROM (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY sak_recip, dte_effective, dte_end ORDER BY CDE_PGM_HEALTH) AS SEQ_PGM_HEALTH
  FROM (
    SELECT DISTINCT
      d01.sak_recip,
      dte_effective,
      dte_end,
      d02.CDE_PGM_HEALTH
    FROM ${catalog}.${schema_name}.${PrevTblNm} d01
    JOIN ${catalog}.${schema_name}.${PrevTblNm2} d02
    ON d01.SAK_RECIP=d02.SAK_RECIP AND d01.SAK_PGM_ELIG=d02.SAK_PGM_ELIG
  ) x
) y
GROUP BY 1,2,3;


In [0]:
%sql
create or replace TEMP VIEW PreviousPartD AS
select distinct
-- Header
 ID_MEDICAID
,ID_MBI_CMS
,t1.REPORT_DTE 
,NUM_CASE
,ENRL_SPAN_TYP
-- Plan Level
,t1.DTE_EFFECTIVE
,t1.DTE_END
,CDE_AID_CATEGORY
,t2.CDE_PGM_HEALTH
from  ${catalog}.${schema_name}.${PrevTblNm} t1
left join  PreviousPartDPMG t2 ON t1.SAK_RECIP = t2.SAK_RECIP  AND t1.DTE_EFFECTIVE = t2.DTE_EFFECTIVE AND t1.DTE_END = t2.DTE_END
;

In [0]:
%sql
select * from PreviousPartD
;

In [0]:

%sql
create or replace table ${catalog}.${schema_name}.${DiffAnlysTblNm} AS 
SELECT 
    -- Common identifiers
     COALESCE(PrevTbl.ID_MEDICAID, CurntTbl.ID_MEDICAID) AS ID_MEDICAID          -- This is the common identifier
    ,COALESCE(PrevTbl.DTE_EFFECTIVE, CurntTbl.DTE_EFFECTIVE) AS DTE_EFFECTIVE -- This is the common identifier
	-- None Compare Fileds
    ,CASE WHEN PrevTbl.REPORT_DTE              != CurntTbl.REPORT_DTE             THEN PrevTbl.REPORT_DTE                  ELSE NULL END AS CurntTbl_REPORT_DTE    -- DERIVED1
    ,CASE WHEN PrevTbl.REPORT_DTE              != CurntTbl.REPORT_DTE             THEN CurntTbl.REPORT_DTE                 ELSE NULL END AS PrevTbl_REPORT_DTE    
    -- Compare Fields
    ,CASE WHEN PrevTbl.ENRL_SPAN_TYP           != CurntTbl.ENRL_SPAN_TYP           THEN PrevTbl.ENRL_SPAN_TYP            ELSE NULL END AS CurntTbl_ENRL_SPAN_TYP    -- Derived2
    ,CASE WHEN PrevTbl.ENRL_SPAN_TYP           != CurntTbl.ENRL_SPAN_TYP           THEN CurntTbl.ENRL_SPAN_TYP           ELSE NULL END AS PrevTbl_ENRL_SPAN_TYP    
    ,CASE WHEN PrevTbl.NUM_CASE                != CurntTbl.NUM_CASE                THEN PrevTbl.NUM_CASE                  ELSE NULL END AS CurntTbl_NUM_CASE                               
    ,CASE WHEN PrevTbl.NUM_CASE                != CurntTbl.NUM_CASE                THEN CurntTbl.NUM_CASE                 ELSE NULL END AS PrevTbl_NUM_CASE                               
    ,CASE WHEN PrevTbl.DTE_END                 != CurntTbl.DTE_END                 THEN PrevTbl.DTE_END                 ELSE NULL END AS CurntTbl_DTE_END                               
    ,CASE WHEN PrevTbl.DTE_END                 != CurntTbl.DTE_END                 THEN CurntTbl.DTE_END                ELSE NULL END AS PrevTbl_DTE_END                               
    ,CASE WHEN PrevTbl.CDE_AID_CATEGORY        != CurntTbl.CDE_AID_CATEGORY        THEN PrevTbl.CDE_AID_CATEGORY        ELSE NULL END AS CurntTbl_CDE_AID_CATEGORY
    ,CASE WHEN PrevTbl.CDE_AID_CATEGORY        != CurntTbl.CDE_AID_CATEGORY        THEN CurntTbl.CDE_AID_CATEGORY       ELSE NULL END AS PrevTbl_CDE_AID_CATEGORY
    ,CASE WHEN PrevTbl.CDE_PGM_HEALTH          != CurntTbl.CDE_PGM_HEALTH          THEN PrevTbl.CDE_PGM_HEALTH          ELSE NULL END AS CurntTbl_CDE_PGM_HEALTH
    ,CASE WHEN PrevTbl.CDE_PGM_HEALTH          != CurntTbl.CDE_PGM_HEALTH          THEN CurntTbl.CDE_PGM_HEALTH         ELSE NULL END AS PrevTbl_CDE_PGM_HEALTH
           FROM CurrentPartD CurntTbl
FULL OUTER JOIN PreviousPartD PrevTbl ON PrevTbl.ID_MEDICAID    = CurntTbl.ID_MEDICAID 
                                     AND PrevTbl.DTE_EFFECTIVE = CurntTbl.DTE_EFFECTIVE
WHERE 
       PrevTbl.NUM_CASE                != CurntTbl.NUM_CASE                  
    OR PrevTbl.ENRL_SPAN_TYP           != CurntTbl.ENRL_SPAN_TYP                     
    OR PrevTbl.DTE_END                 != CurntTbl.DTE_END                
    OR PrevTbl.CDE_AID_CATEGORY        != CurntTbl.CDE_AID_CATEGORY                   
    OR PrevTbl.CDE_PGM_HEALTH          != CurntTbl.CDE_PGM_HEALTH
   AND PrevTbl.DTE_EFFECTIVE is not null
;

In [0]:
%sql
 select count(*) from ${catalog}.${schema_name}.${DiffAnlysTblNm};
 

In [0]:
%sql
 select * from ${catalog}.${schema_name}.${DiffAnlysTblNm};

In [0]:
%sql
WITH
  Sum_Total AS (
    SELECT count(*) cntTotal
    FROM ${catalog}.${schema_name}.${DiffAnlysTblNm}
    WHERE TRUE 
  ),
  Cnt_ENRL_SPAN_TYP AS (
    SELECT COUNT(*) AS Diff_ENRL_SPAN_TYP
    FROM ${catalog}.${schema_name}.${DiffAnlysTblNm}
    WHERE CurntTbl_ENRL_SPAN_TYP IS NOT NULL
       OR PrevTbl_ENRL_SPAN_TYP IS NOT NULL
  ),
  Cnt_NUM_CASE AS (
    SELECT COUNT(*) AS Diff_NUM_CASE
    FROM ${catalog}.${schema_name}.${DiffAnlysTblNm}
    WHERE CurntTbl_NUM_CASE IS NOT NULL
       OR PrevTbl_NUM_CASE IS NOT NULL
  ),
  Cnt_DTE_END AS (
    SELECT COUNT(*) AS Diff_DTE_END
    FROM ${catalog}.${schema_name}.${DiffAnlysTblNm}
    WHERE CurntTbl_DTE_END IS NOT NULL
       OR PrevTbl_DTE_END IS NOT NULL
  ),
  Cnt_CDE_AID_CATEGORY AS (
    SELECT COUNT(*) AS Diff_CDE_AID_CATEGORY
    FROM ${catalog}.${schema_name}.${DiffAnlysTblNm}
    WHERE CurntTbl_CDE_AID_CATEGORY IS NOT NULL
       OR PrevTbl_CDE_AID_CATEGORY IS NOT NULL
  ),
  Cnt_CDE_PGM_HEALTH AS (
    SELECT COUNT(*) AS Diff_CDE_PGM_HEALTH
    FROM ${catalog}.${schema_name}.${DiffAnlysTblNm}
    WHERE CurntTbl_CDE_PGM_HEALTH IS NOT NULL
       OR PrevTbl_CDE_PGM_HEALTH IS NOT NULL
  )

SELECT
  format_number(t6.cntTotal, 0)                             AS cntTotal,
  format_number(t1.Diff_ENRL_SPAN_TYP, 0)                   AS Diff_ENRL_SPAN_TYP,
  format_number(CASE WHEN t6.cntTotal = 0 THEN 0
                  ELSE t1.Diff_ENRL_SPAN_TYP * 100 / t6.cntTotal
             END, '###.#')                                  AS Pcntg_ENRL_SPAN_TYP,
  format_number(t2.Diff_NUM_CASE, 0)                        AS Diff_NUM_CASE,
  format_number(CASE WHEN t6.cntTotal = 0 THEN 0
                  ELSE t2.Diff_NUM_CASE * 100 / t6.cntTotal
             END, '###.#')                                  AS Pcntg_NUM_CASE,
  format_number(t3.Diff_DTE_END, 0)                       AS Diff_DTE_END,
  format_number(CASE WHEN t6.cntTotal = 0 THEN 0
                  ELSE t3.Diff_DTE_END * 100 / t6.cntTotal
             END, '###.#')                                  AS Pcntg_DTE_END,
  format_number(t4.Diff_CDE_AID_CATEGORY, 0)              AS Diff_CDE_AID_CATEGORY,
  format_number(CASE WHEN t6.cntTotal = 0 THEN 0
                  ELSE t4.Diff_CDE_AID_CATEGORY * 100 / t6.cntTotal
             END, '###.#')                                  AS Pcntg_CDE_AID_CATEGORY,
  format_number(t5.Diff_CDE_PGM_HEALTH, 0)                AS Diff_CDE_PGM_HEALTH,
  format_number(CASE WHEN t6.cntTotal = 0 THEN 0
                  ELSE t5.Diff_CDE_PGM_HEALTH * 100 / t6.cntTotal
             END, '###.#')                                  AS Pcntg_CDE_PGM_HEALTH
FROM
  Cnt_ENRL_SPAN_TYP t1,
  Cnt_NUM_CASE t2,
  Cnt_DTE_END t3,
  Cnt_CDE_AID_CATEGORY t4,
  Cnt_CDE_PGM_HEALTH t5,
  Sum_Total t6
;

In [0]:
%sql
WITH
  Match_Counts AS (
    SELECT DISTINCT PrevTbl.ID_MEDICAID
                   ,PrevTbl.DTE_EFFECTIVE 
          FROM CurrentPartD CurntTbl
          JOIN PreviousPartD PrevTbl ON PrevTbl.ID_MEDICAID    = CurntTbl.ID_MEDICAID 
                                    AND PrevTbl.DTE_EFFECTIVE = CurntTbl.DTE_EFFECTIVE
    WHERE TRUE  
      AND PrevTbl.DTE_EFFECTIVE is not null
  ),
  Get_CntsNew_VsOld AS (
    SELECT COUNT(*) AS CntNew_VsOld
    FROM CurrentPartD CurntTbl
    WHERE TRUE
      AND CurntTbl.DTE_EFFECTIVE is not null
      AND NOT EXISTS
       (
        SELECT 1
          FROM Match_Counts t1
         WHERE TRUE 
           AND CurntTbl.ID_MEDICAID    = t1.ID_MEDICAID
           AND CurntTbl.DTE_EFFECTIVE = t1.DTE_EFFECTIVE
       )
  ),
  Get_CntsOld_VsNew AS (
    SELECT COUNT(*) AS CntOld_VsNew
    FROM PreviousPartD PrevTbl
    WHERE TRUE
       AND PrevTbl.DTE_EFFECTIVE is not null
      AND NOT EXISTS
       (
        SELECT 1
          FROM Match_Counts t4
         WHERE TRUE 
           AND PrevTbl.ID_MEDICAID    = t4.ID_MEDICAID
           AND PrevTbl.DTE_EFFECTIVE = t4.DTE_EFFECTIVE
       )
  )
SELECT DISTINCT format_number((SELECT COUNT(*) FROM Match_Counts), 0) AS TotalMatch
     , format_number(t2.CntNew_VsOld, 0) AS RecordsNewVsOld
     , format_number(t3.CntOld_VsNew, 0) AS RecordsOldVsNew
  FROM Match_Counts t1
     , Get_CntsNew_VsOld t2
     , Get_CntsOld_VsNew t3
;

Databricks data profile. Run in Databricks to view.

In [0]:
sql_out = spark.sql(f"""
WITH
  Recip_Counts AS (
    SELECT COUNT( DISTINCT ID_MEDICAID) AS RecipCnt
          FROM {catalog}.{schema_name}.{CurntTblNm}
    WHERE TRUE
  ),
  EligRecip_Counts AS (
    SELECT COUNT(*)   AS ProvMcaidCnt
         , SUM(IDCnt) AS McaidCnt
    FROM 
    (SELECT DISTINCT CDE_AID_CATEGORY
          , COUNT(DISTINCT ID_MEDICAID) AS IDCnt
       FROM {catalog}.{schema_name}.{CurntTblNm}
      WHERE TRUE
        AND current_date() BETWEEN DTE_EFFECTIVE AND DTE_END
      GROUP BY 1
    )
  ),
  AidCtgry_Counts AS (
    SELECT COUNT(*)   AS AidCtgryCnt
         , SUM(IDCnt) AS McaidCnt
    FROM 
    (SELECT DISTINCT CDE_AID_CATEGORY
          , COUNT(DISTINCT ID_MEDICAID) AS IDCnt
       FROM {catalog}.{schema_name}.{CurntTblNm}
      GROUP BY 1
    )
  )
--
-- Main SQL
--
SELECT format_number(t1.RecipCnt, 0) AS TotalRecip
     , format_number(t2.AidCtgryCnt, 0) AS TotalAidCtgry
     , format_number(t2.McaidCnt, 0) AS TotalMcaid
     , format_number(t3.ProvMcaidCnt, 0) AS TotalEligProvMcaid
     , format_number(t3.McaidCnt, 0) AS TotalEligMcaid
  FROM Recip_Counts t1
     , AidCtgry_Counts t2
     , EligRecip_Counts t3
;
               """)
display(sql_out)